<a href="https://colab.research.google.com/github/Abhi-pan1982/ML_NLP/blob/main/Realtime_Audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Developing Real-time Speech-to-Text

Real-time speech-to-text (STT) involves converting spoken audio into text as it's being spoken. This is a complex task that typically requires a combination of audio processing, machine learning models, and efficient streaming. Here's a general approach and some common technologies:

### 1. Audio Input and Preprocessing

*   **Microphone Input:** Capture audio from a microphone or an audio stream.
*   **Audio Libraries:** Use libraries like `PyAudio` (for Python) or web browser APIs (for JavaScript) to manage audio input.
*   **Preprocessing:**
    *   **Noise Reduction:** Filter out background noise.
    *   **Silence Detection:** Identify and remove periods of silence to focus on speech.
    *   **Resampling/Normalization:** Ensure audio is at a consistent sample rate and volume for the STT model.
    *   **Framing:** Divide the audio into small, overlapping frames for feature extraction.

### 2. Feature Extraction

*   **Spectrograms/MFCCs:** Convert the raw audio frames into features that are more meaningful for machine learning models. Mel-frequency cepstral coefficients (MFCCs) and spectrograms are common choices.

### 3. Acoustic Model (AM)

*   **Purpose:** The Acoustic Model is trained to map these audio features to phonemes (basic units of sound) or directly to sub-word units (like graphemes or senones).
*   **Technologies:**
    *   **Deep Neural Networks (DNNs):** Recurrent Neural Networks (RNNs, especially LSTMs and GRUs), Convolutional Neural Networks (CNNs), and Transformer networks are frequently used.
    *   **Hidden Markov Models (HMMs) with GMMs:** Traditional approach, still used in some systems.
    *   **CTC (Connectionist Temporal Classification):** A popular loss function used with RNNs to allow direct mapping from input sequence to output sequence without explicit alignment.

### 4. Pronunciation Lexicon

*   **Purpose:** A dictionary that maps words to their phonetic pronunciations (sequences of phonemes).

### 5. Language Model (LM)

*   **Purpose:** Helps to predict the next word in a sequence based on the preceding words. It improves accuracy by favoring grammatically correct and contextually appropriate word sequences.
*   **Technologies:**
    *   **N-gram Models:** Statistical models that predict word likelihood based on the previous N-1 words.
    *   **Neural Language Models:** RNNs (LSTMs, GRUs) and Transformer models (like BERT, GPT) are used for more sophisticated language modeling.

### 6. Decoder / Search Algorithm

*   **Purpose:** Combines the outputs of the Acoustic Model, Pronunciation Lexicon, and Language Model to find the most probable sequence of words.
*   **Algorithms:**
    *   **Viterbi Algorithm:** Common for HMM-based systems.
    *   **Beam Search:** Widely used with neural network models to explore a limited set of the most promising paths through the possible word sequences.

### 7. Real-time Considerations

*   **Streaming:** Process audio chunks incrementally as they arrive, rather than waiting for the entire audio segment.
*   **Low Latency:** Optimize models and decoding algorithms for speed.
*   **Endpointing:** Automatically detect when a user has finished speaking a phrase or sentence.
*   **Incremental Decoding:** Generate partial transcripts that are updated as more audio is received.

### Popular STT Services and Frameworks:

*   **Cloud-based APIs:**
    *   **Google Cloud Speech-to-Text:** Highly accurate, supports many languages, good for real-time and batch.
    *   **AWS Transcribe:** Amazon's offering, similar capabilities.
    *   **Azure Speech Service:** Microsoft's comprehensive speech platform.
    *   **OpenAI's Whisper:** A powerful open-source model that has gained significant traction for its accuracy and multi-language support.
*   **Open-source Libraries/Frameworks (for self-hosting/custom models):**
    *   **Kaldi:** A very powerful and flexible toolkit for speech recognition research and development.
    *   **Mozilla DeepSpeech:** An open-source STT engine based on Baidu's Deep Speech research.
    *   **CMU Sphinx (PocketSphinx):** A lightweight, embedded speech recognition engine.
    *   **Hugging Face Transformers (with Wav2Vec2, HuBERT, etc.):** Provides pre-trained models that can be fine-tuned for STT tasks.
    *   **AssemblyAI:** Offers a powerful API for STT with advanced features like speaker diarization and summarization.

### Example Workflow (Conceptual):

1.  **Client-side (e.g., Web browser or Python script):**
    *   Capture audio from the microphone.
    *   Stream small chunks of audio (e.g., 100ms) to a server or directly to a cloud API.

2.  **Server-side (if self-hosting an STT model):**
    *   Receive audio chunks.
    *   Preprocess audio (noise reduction, framing).
    *   Extract features (MFCCs).
    *   Feed features to the Acoustic Model.
    *   Use a decoder with a Language Model and Pronunciation Lexicon to generate a transcript.
    *   Stream back the incremental transcript to the client.

3.  **Cloud API:**
    *   Simply send the audio stream to the API.
    *   The API handles all the processing and streams back the recognized text.

For most applications requiring high accuracy and scalability, using a cloud-based STT API like Google Cloud Speech-to-Text or OpenAI's Whisper API is the most straightforward and effective solution.

## OpenAI Whisper API Example

This example demonstrates how to use the OpenAI Whisper API for transcribing an audio file. You will need an OpenAI API key for this.

First, we'll install the `openai` library. Then, for demonstration purposes, we'll create a simple dummy audio file, and finally, show the transcription process.

In [ ]:
import sys

# Install the openai library
!{sys.executable} -m pip install openai

print('OpenAI library installed successfully.')

For this example, we'll create a dummy audio file using `pydub` and `numpy`. If you have your own audio file, you can skip this step and use your file instead.

In [ ]:
import numpy as np
from pydub import AudioSegment
from pydub.playback import play # Optional: for playing the generated audio
import os

# Install pydub if not already installed
!{sys.executable} -m pip install pydub

# Create a dummy audio file for demonstration
def generate_dummy_audio(filename='dummy_audio.mp3', duration_seconds=5, sample_rate=44100):
    # Generate a simple sine wave
    frequency = 440  # Hz (A4 note)
    t = np.linspace(0, duration_seconds, int(sample_rate * duration_seconds), endpoint=False)
    amplitude = np.iinfo(np.int16).max * 0.5
    data = amplitude * np.sin(2 * np.pi * frequency * t)
    data = data.astype(np.int16)

    # Create an AudioSegment from the numpy array
    audio_segment = AudioSegment(data.tobytes(), frame_rate=sample_rate, sample_width=data.dtype.itemsize, channels=1)

    # Export to MP3 (requires ffmpeg or libav to be installed on the system)
    # In Colab, ffmpeg is usually pre-installed.
    audio_segment.export(filename, format="mp3")
    print(f"Dummy audio file '{filename}' created with a duration of {duration_seconds} seconds.")
    return filename

# Generate the audio file
audio_file_path = generate_dummy_audio()

# Optional: Play the generated audio (requires simpleaudio or other playback library)
# You might need to install `simpleaudio` if you want to play it back in a local environment
# !pip install simpleaudio
# audio = AudioSegment.from_mp3(audio_file_path)
# play(audio)


Now, let's use the OpenAI Whisper model to transcribe the audio. You'll need to set up your OpenAI API key in Colab's secrets manager (named `OPENAI_API_KEY`) for secure access.

If you don't have an API key, you can get one from the [OpenAI website](https://platform.openai.com/account/api-keys).

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Get your OpenAI API key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Open the audio file in binary read mode
with open(audio_file_path, "rb") as audio_file:
    # Call the Whisper API to transcribe the audio
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="text"
    )

print("Transcription:")
print(transcript)

## Real-time Audio Capture and Transcription using Whisper

For real-time transcription with a service like OpenAI's Whisper, which typically processes complete audio files, we need to implement a chunking strategy:

1.  **Capture Audio:** Record short segments (chunks) of audio from the microphone.
2.  **Save Chunk:** Save each audio chunk to a temporary file.
3.  **Transcribe Chunk:** Send the temporary file to the Whisper API for transcription.
4.  **Display Transcription:** Print the transcription and repeat.

This method introduces latency, as each chunk needs to be recorded and then processed by the API. For true low-latency real-time STT, a dedicated streaming API or a locally running streaming model would be more appropriate, but this approach works well for simulating real-time interaction with a file-based API.

In [ ]:
import sys

# Install necessary libraries for audio recording
!{sys.executable} -m pip install sounddevice soundfile

print('`sounddevice` and `soundfile` libraries installed successfully.')

In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import os
import time
from openai import OpenAI
from google.colab import userdata

# Configuration
SAMPLERATE = 16000  # Sample rate for audio capture (Hz)
CHANNELS = 1        # Mono audio
CHUNK_DURATION = 5  # seconds: duration of each audio chunk to record
TEMP_AUDIO_FILE = 'realtime_audio_chunk.wav'

# Get your OpenAI API key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

print(f"Starting real-time audio capture. Recording in {CHUNK_DURATION}-second chunks.")
print("Press Ctrl+C in this cell's output to stop the entire process.")

try:
    while True:
        print("\nRecording new chunk...")
        # Record audio for CHUNK_DURATION seconds
        audio_data = sd.rec(int(CHUNK_DURATION * SAMPLERATE), samplerate=SAMPLERATE, channels=CHANNELS, dtype='int16')
        sd.wait()  # Wait until recording is finished for this chunk

        print(f"Chunk recording finished. Saving to {TEMP_AUDIO_FILE} and starting transcription...")
        # Save the recorded chunk to a temporary WAV file
        sf.write(TEMP_AUDIO_FILE, audio_data, SAMPLERATE)

        # Transcribe the audio chunk using OpenAI Whisper
        try:
            with open(TEMP_AUDIO_FILE, "rb") as audio_file:
                transcript = client.audio.transcriptions.create(
                    model="whisper-1",
                    file=audio_file,
                    response_format="text"
                )
            print(f"Transcription: {transcript.strip()}")
        except Exception as e:
            print(f"Error transcribing chunk: {e}")

        # Clean up the temporary file (optional, depends on debugging needs)
        if os.path.exists(TEMP_AUDIO_FILE):
            os.remove(TEMP_AUDIO_FILE)

except KeyboardInterrupt:
    print("\nReal-time transcription stopped by user.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    # Ensure any temporary file is removed on exit
    if os.path.exists(TEMP_AUDIO_FILE):
        os.remove(TEMP_AUDIO_FILE)


### Important Considerations:

*   **Microphone Access:** When you run the `sounddevice` code in Colab for the first time, your browser will likely ask for permission to access your microphone. You must grant this permission for the code to work.
*   **Latency:** As mentioned, this method involves recording, saving, and then sending to an API, which introduces noticeable latency. For extremely low-latency applications, you might need to explore continuous streaming APIs (if available for Whisper or other services) or local models optimized for real-time inference.
*   **Cost:** Each API call to Whisper incurs a cost. Be mindful of your usage when running this in a loop.
*   **Stopping the Process:** You will need to manually stop the execution of the Python cell (e.g., by clicking the 'Stop' button next to the cell or pressing `Ctrl+C` in the cell's output) to end the continuous recording and transcription loop.